In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import glob
import os
import string
import random
import time

from matplotlib.ticker import FormatStrFormatter
from matplotlib.patches import Patch
from pathlib import Path
from scipy import io
import nmi
from helps import *
fig, axs = plt.subplots(1, 2, figsize=(14, 1))
axs[0].bar(range(len(colors_to_use)), [1] * len(colors_to_use), color=colors_to_use)
axs[0].set_xticks(range(len(colors_to_use)))
axs[1].bar(range(len(colors_to_use_pastel)), [1] * len(colors_to_use_pastel), color=colors_to_use_pastel)
axs[1].set_xticks(range(len(colors_to_use_pastel)))
plt.show()

# Figure 2

In [ ]:
palette={alg : colors_to_use_pastel[i] for i,alg in enumerate(["ShareTopic","Mowgli","bionSBM","MultiVI","totalVI"])}
palette["COBOLT"]=colors_to_use_pastel[8]
params = {
    'axes.labelsize': 17,
    'axes.titlesize': 17,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    "lines.linewidth": 3,
    "figure.dpi": 300}
plt.rcParams.update(params)
fig, axs = plt.subplots(2,3, figsize=(16,10))
axs=axs.flatten()

for l, dataset in enumerate(["PBMC","HSPC","BMMCMultiOme","MouseSkin","BMMCCite","Spleen"]):
    if dataset in ["BMMCCite", "Spleen"]:
        order=["ADT+GEX", "ADT+mRNA+lncRNA", "ADT+mRNA", "ADT+lncRNA", "mRNA+lncRNA"][::-1]
    else:
        order=["Peak+GEX", "Peak+mRNA", "Peak+lncRNA","Peak+mRNA+lncRNA", "mRNA+lncRNA"][::-1]

    df_best=create_results(dataset, "CellType")[0].replace({"_" : "+"}, regex=True)
    df_best["log10(NMI/NMI*)"]=np.log10(df_best["NMI/NMI*"])
    df_best=df_best[df_best["Algorithm"]!="MOFA+"]
    
    for i, exp in enumerate(palette.keys()):
        d = df_best[df_best["Algorithm"] == exp]
        means = d.groupby("Exp")["log10(NMI/NMI*)"].mean()
        sems = 3 * d.groupby("Exp")["log10(NMI/NMI*)"].sem()
        new_order_temp=["+".join(o.split("+")[1:]) for o in list(d.groupby("Exp")["log10(NMI/NMI*)"].mean().index)]
        means.index=new_order_temp
        sems.index=new_order_temp
        exps_present = [o for o in order if o in means.index]
        
        y_positions = np.array([order.index(o) for o in exps_present], dtype=float)
        
        # Add small vertical offset per method to avoid overlap
        offset = (i - (len(palette) - 1) / 2) * 0.1
        y_positions += offset
        axs[l].errorbar(x=means[exps_present], y=y_positions, xerr=sems[exps_present], ls='none', ecolor=palette[exp], 
                        elinewidth=5, marker="o", c=palette[exp], label=exp)

    axs[l].set_title(dataset, y=1.05)
    axs[l].set_xlim([0, 3])
    xlabels=["\n".join(o.split("+")) for o in order]
    axs[l].set_yticks(range(len(order)))
    axs[l].set_yticklabels(xlabels)
    axs[l].spines['top'].set_visible(False)
    axs[l].spines['right'].set_visible(False)  
    axs[l].spines['bottom'].set_visible(True)
    axs[l].spines['left'].set_visible(True)  
    axs[l].set_xlabel("NMI/NMI* (log10)")
    axs[l].set_ylabel("Experiment")
    axs[l].annotate(string.ascii_uppercase[l],xy=(-0.25, 1.1),xycoords='axes fraction',fontsize=20,fontweight="bold",ha='left')

handles, labels = axs[2].get_legend_handles_labels()
desired_order = ["bionSBM", "Mowgli", "ShareTopic","MultiVI","totalVI","COBOLT"]
order_idx = [labels.index(m) for m in desired_order]
handles = [handles[i] for i in order_idx]
labels = [labels[i] for i in order_idx]
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=1, title="Method", fontsize=14, title_fontsize=16, frameon=False)
fig.tight_layout(h_pad=3)
plt.savefig(f"Figures/bionSBM_Fig2.png", format="png", dpi=300, bbox_inches='tight')
plt.show()